# ITRI external validation: classical methods

Choose Runtime → Run all. The notebook installs dependencies and prepares readable Python source. When the upload prompt appears, select your ITRI_external_PRIVATE_data_snapshot.zip. The archive hash is checked before any experiment. The full run then proceeds and automatically downloads the results ZIP.

Use a CPU runtime. Keep the input snapshot private. The downloaded results contain metrics, code, protocol and metadata but no raw OHLCV. Long source cells are collapsed; expand them to inspect ordinary Python code. The neural BRITS/SAITS stage remains separate.

In [ ]:
%pip -q install numpy==2.1.3 pandas==2.2.3 scipy==1.18.1 scikit-learn==1.6.1

In [ ]:
from pathlib import Path
work_dir = Path('/content/itri_classical')
work_dir.mkdir(parents=True, exist_ok=True)
print('Working directory ready:', work_dir)

In [ ]:
%%writefile /content/itri_classical/masking.py
"""Controlled missing-run masks for a new, separately reported validation study.

The archived ITRI experiment permits adjacent sampled segments. This module
does not alter those historical masks or results. It defines strict runs for a
future experiment in which the requested block length has a literal meaning.
"""

from __future__ import annotations

import numpy as np


def run_lengths(mask: np.ndarray) -> np.ndarray:
    """Return every maximal missing-run length, separately for each feature."""
    if mask.ndim != 2 or mask.dtype != np.bool_:
        raise ValueError("mask must be a two-dimensional boolean array")
    lengths: list[int] = []
    for feature in range(mask.shape[1]):
        edges = np.flatnonzero(np.diff(np.r_[False, mask[:, feature], False]))
        lengths.extend((edges[1::2] - edges[::2]).tolist())
    return np.asarray(lengths, dtype=int)


def strict_block_mask(
    shape: tuple[int, int], rate: float, block_length: int, seed: int,
    edge_guard: int = 10,
) -> np.ndarray:
    """Place nonadjacent runs of length ``block_length`` at an exact cell rate.

    Every feature receives one full run before the remaining runs are sampled,
    so feature-wise errors are defined for every condition. A final shorter
    run is allowed only when the rounded target cell count is not divisible
    by the requested block length. One observed row separates every pair of
    runs within a feature. Other features may be missing on the same dates.
    """
    rows, features = shape
    if rows <= 2 * edge_guard + block_length or features < 1:
        raise ValueError("not enough rows/features for the block and guard")
    if not 0 < rate < 1 or block_length < 1 or edge_guard < 0:
        raise ValueError("invalid rate, block length, or edge guard")
    eligible_rows = rows - 2 * edge_guard
    target = max(1, round(rate * eligible_rows * features))
    if target < features * block_length:
        raise ValueError("target too small for one full masked run per feature")
    rng = np.random.default_rng(seed)
    mask = np.zeros(shape, dtype=bool)
    # The endpoint is a five-feature macro mean, so a zero-mask feature is
    # invalid. Reserve exactly one nominal-length run per feature first.
    for feature in rng.permutation(features):
        start = int(rng.integers(edge_guard,
                                 rows - edge_guard - block_length + 1))
        mask[start:start + block_length, feature] = True
    remaining = target - features * block_length
    attempts = 0
    max_attempts = 1000 * target

    while remaining:
        length = min(block_length, remaining)
        feature = int(rng.integers(features))
        start = int(rng.integers(edge_guard, rows - edge_guard - length + 1))
        neighborhood = mask[max(edge_guard, start - 1):
                            min(rows - edge_guard, start + length + 1), feature]
        if not neighborhood.any():
            mask[start:start + length, feature] = True
            remaining -= length
        attempts += 1
        if attempts >= max_attempts:
            raise RuntimeError("could not place the target runs without adjacency")

    lengths = run_lengths(mask)
    if (int(mask.sum()) != target or lengths.max() > block_length or
            not np.all(mask.sum(axis=0) >= block_length)):
        raise AssertionError("strict-block mask invariant failed")
    if np.any(mask[:edge_guard]) or np.any(mask[rows - edge_guard:]):
        raise AssertionError("strict-block mask crossed the edge guard")
    return mask


In [ ]:
%%writefile /content/itri_classical/run_classical_benchmark.py
#!/usr/bin/env python3
"""Frozen OHLCV external-validation benchmark, classical comparators.

This program reads the PRIVATE vendor snapshot ZIP without extracting it. It
never writes OHLCV values to the public result directory. All methods receive
the exact same mask per asset/fold/condition. Outputs are checkpointed by
asset/fold. Use --pilot for a small smoke test; never report pilot results as
the full prespecified experiment.
"""

from __future__ import annotations

import argparse
import hashlib
import io
import json
import os
import platform
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.interpolate import PchipInterpolator
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.exceptions import ConvergenceWarning

from masking import run_lengths, strict_block_mask


FEATURES = ("open", "high", "low", "close", "volume")
METHODS = ("linear", "pchip", "knn5", "chained_ridge", "itri_w1_p5")
FOLDS = {
    "A": {"train": (2015, 2018), "val": (2019, 2019), "test": (2020, 2021)},
    "B": {"train": (2015, 2020), "val": (2021, 2021), "test": (2022, 2023)},
    "C": {"train": (2015, 2022), "val": (2023, 2023), "test": (2024, 2025)},
}
BLOCKS = (1, 2, 3, 5, 10, 20)
RATES = (0.10, 0.20, 0.30)
SEEDS = (11, 23, 37)
EDGE_GUARD = 20


def sha256(blob: bytes) -> str:
    return hashlib.sha256(blob).hexdigest()


def stable_mask_seed(manifest_sha: str, ticker: str, fold: str,
                     partition: str, block: int, rate: float,
                     repetition_seed: int) -> int:
    key = (f"{manifest_sha}|{ticker}|{fold}|{partition}|"
           f"{block}|{rate:.2f}|{repetition_seed}")
    return int.from_bytes(hashlib.sha256(key.encode()).digest()[:4], "big")


def split_frame(frame: pd.DataFrame, fold: str) -> dict[str, pd.DataFrame]:
    parts = {}
    for name, (start_year, end_year) in FOLDS[fold].items():
        subset = frame.loc[(frame.index.year >= start_year) &
                           (frame.index.year <= end_year), list(FEATURES)]
        if subset.empty:
            raise ValueError(f"empty {fold}/{name} partition")
        parts[name] = subset.copy()
    return parts


def train_observed_scaler(train: np.ndarray, hidden: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if train.shape != hidden.shape:
        raise ValueError("training array/mask shape mismatch")
    means, scales = [], []
    for feature in range(train.shape[1]):
        observed = train[~hidden[:, feature], feature]
        if observed.size < 2 or not np.isfinite(observed).all():
            raise ValueError(f"insufficient finite training data for feature {feature}")
        mean = float(observed.mean())
        scale = float(observed.std(ddof=0))
        if scale <= 1e-12:
            raise ValueError(f"near-zero training-observed std for feature {feature}")
        means.append(mean)
        scales.append(scale)
    return np.asarray(means), np.asarray(scales)


def corrupt(values: np.ndarray, hidden: np.ndarray) -> np.ndarray:
    output = values.copy()
    output[hidden] = np.nan
    return output


def linear_fill(corrupt_values: np.ndarray) -> np.ndarray:
    return pd.DataFrame(corrupt_values).interpolate(
        method="linear", limit_direction="both").ffill().bfill().to_numpy(float)


def pchip_fill(corrupt_values: np.ndarray) -> np.ndarray:
    out = corrupt_values.copy()
    x = np.arange(len(out))
    for feature in range(out.shape[1]):
        seen = np.isfinite(out[:, feature])
        if seen.sum() < 2:
            raise ValueError("PCHIP needs at least two observed cells")
        curve = PchipInterpolator(x[seen], out[seen, feature], extrapolate=True)
        out[~seen, feature] = curve(x[~seen])
    return out


class ITRIImputer:
    """Exact w=1, p=5 temporal-regression logic from the archived implementation.

    Only the panel and observed-only scaling differ from the original study.
    The base model has not been modified to chase external test results.
    """

    def __init__(self) -> None:
        self.window = 1
        self.models_: list[LinearRegression | None] = []

    def temporal_matrix(self, col: np.ndarray) -> np.ndarray:
        z = np.column_stack((np.roll(col, 1), np.roll(col, -1)))
        z[0, :] = np.nan
        z[-1, :] = np.nan
        return z

    def fit(self, train_corrupt: np.ndarray) -> "ITRIImputer":
        missing = np.isnan(train_corrupt)
        completed = linear_fill(train_corrupt)
        self.models_ = []
        for feature in range(train_corrupt.shape[1]):
            z = self.temporal_matrix(completed[:, feature])
            good = (~missing[:, feature]) & np.isfinite(z).all(axis=1)
            if int(good.sum()) < 4:
                self.models_.append(None)
                continue
            model = LinearRegression()
            model.fit(z[good], completed[good, feature])
            self.models_.append(model)
        return self

    def transform(self, test_corrupt: np.ndarray) -> np.ndarray:
        missing = np.isnan(test_corrupt)
        observed = ~missing
        completed = linear_fill(test_corrupt)
        for _ in range(5):
            for feature, model in enumerate(self.models_):
                if model is None:
                    continue
                z = self.temporal_matrix(completed[:, feature])
                rows = missing[:, feature] & np.isfinite(z).all(axis=1)
                if rows.any():
                    completed[rows, feature] = model.predict(z[rows])
            completed[observed] = test_corrupt[observed]
        return completed


def run_methods(train_corrupt: np.ndarray, test_corrupt: np.ndarray,
                random_seed: int) -> dict[str, tuple[np.ndarray, float, int]]:
    outputs = {}
    start = time.perf_counter()
    outputs["linear"] = (linear_fill(test_corrupt), time.perf_counter() - start, 0)
    start = time.perf_counter()
    outputs["pchip"] = (pchip_fill(test_corrupt), time.perf_counter() - start, 0)
    start = time.perf_counter()
    knn = KNNImputer(n_neighbors=5, weights="distance")
    knn.fit(train_corrupt)
    outputs["knn5"] = (knn.transform(test_corrupt), time.perf_counter() - start, 0)
    start = time.perf_counter()
    chained = IterativeImputer(
        estimator=Ridge(alpha=1.0), max_iter=10, tol=1e-3,
        initial_strategy="mean", imputation_order="ascending",
        sample_posterior=False, random_state=random_seed,
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        chained.fit(train_corrupt)
        chained_estimate = chained.transform(test_corrupt)
    nonconvergence = sum(issubclass(note.category, ConvergenceWarning)
                         for note in caught)
    outputs["chained_ridge"] = (chained_estimate,
                                 time.perf_counter() - start, nonconvergence)
    start = time.perf_counter()
    temporal = ITRIImputer().fit(train_corrupt)
    outputs["itri_w1_p5"] = (temporal.transform(test_corrupt),
                               time.perf_counter() - start, 0)
    if tuple(outputs) != METHODS:
        raise AssertionError("method registry/order mismatch")
    for method, (estimate, _, _) in outputs.items():
        if estimate.shape != test_corrupt.shape or not np.isfinite(estimate).all():
            raise ValueError(f"{method} produced invalid estimates")
        observed = np.isfinite(test_corrupt)
        if not np.allclose(estimate[observed], test_corrupt[observed],
                           atol=1e-9, rtol=1e-9):
            raise ValueError(f"{method} changed originally observed cells")
    return outputs


def score_features(truth_raw: np.ndarray, estimate_scaled: np.ndarray,
                   hidden: np.ndarray, mean: np.ndarray, scale: np.ndarray,
                   common: dict, elapsed: float,
                   convergence_warning_count: int) -> list[dict]:
    estimate_raw = estimate_scaled * scale + mean
    results = []
    for feature_index, feature in enumerate(FEATURES):
        selected = hidden[:, feature_index]
        count = int(selected.sum())
        if count < 1:
            raise ValueError(f"no masked cells for {feature}")
        error = estimate_raw[selected, feature_index] - truth_raw[selected, feature_index]
        mae = float(np.abs(error).mean())
        rmse = float(np.sqrt(np.square(error).mean()))
        results.append({
            **common,
            "feature": feature,
            "masked_cells": count,
            "mae_raw": mae,
            "rmse_raw": rmse,
            "mae_trainstd": mae / scale[feature_index],
            "rmse_trainstd": rmse / scale[feature_index],
            "fit_and_impute_seconds": elapsed,
            "convergence_warning_count": convergence_warning_count,
        })
    return results


def load_verified_asset(archive: zipfile.ZipFile, record: dict) -> pd.DataFrame:
    name = f"clean_vendor_cache_do_not_publish/{record['ticker']}_2015-01-01_2026-01-01.csv"
    blob = archive.read(name)
    if sha256(blob) != record["clean_sha256"]:
        raise ValueError(f"private data hash mismatch: {record['ticker']}")
    frame = pd.read_csv(io.BytesIO(blob), index_col=0,
                        parse_dates=True, float_precision="round_trip")
    if frame.columns.tolist() != list(FEATURES) or len(frame) != record["row_count"]:
        raise ValueError(f"private data shape mismatch: {record['ticker']}")
    if not np.isfinite(frame.to_numpy(float)).all():
        raise ValueError(f"nonfinite data: {record['ticker']}")
    return frame


def run(snapshot: Path, output_dir: Path, pilot: bool = False) -> None:
    with zipfile.ZipFile(snapshot) as archive:
        manifest_blob = archive.read("asset_universe_v1.json")
        receipt_blob = archive.read("acquisition_qc_receipt.json")
        receipt = json.loads(receipt_blob)
        manifest = json.loads(manifest_blob)
        manifest_sha = sha256(manifest_blob)
        if receipt["manifest_sha256"] != manifest_sha:
            raise ValueError("snapshot manifest/receipt SHA mismatch")
        if receipt["eligible_count"] != 34 or receipt["excluded_count"]:
            raise ValueError("unexpected eligibility count; revise only by protocol amendment")
        if [x["ticker"] for x in manifest["assets"]] != [
                x["ticker"] for x in receipt["assets"]]:
            raise ValueError("manifest/receipt asset order mismatch")

        output_dir.mkdir(parents=True, exist_ok=True)
        shard_dir = output_dir / "shards"
        shard_dir.mkdir(exist_ok=True)
        versions = {"python": platform.python_version(), "numpy": np.__version__,
                    "pandas": pd.__version__, "scipy": scipy.__version__,
                    "sklearn": sklearn.__version__}
        run_fingerprint = sha256(
            Path(__file__).read_bytes() + Path(__file__).with_name("masking.py").read_bytes()
            + manifest_blob + receipt_blob + str(pilot).encode()
            + json.dumps(versions, sort_keys=True).encode())
        run_meta = {
            "run_fingerprint": run_fingerprint,
            "protocol": manifest["protocol_version"],
            "manifest_sha256": manifest_sha,
            "private_zip_sha256": sha256(snapshot.read_bytes()),
            "pilot_not_publishable": pilot,
            "methods": METHODS,
            "folds": FOLDS,
            "blocks": BLOCKS,
            "rates": RATES,
            "seeds": SEEDS,
            "edge_guard": EDGE_GUARD,
            "primary_panel": FEATURES,
            "primary_endpoint": "macro mean across 5 feature-wise RMSE/training-observed-SD",
            "versions": versions,
        }
        (output_dir / "run_metadata.json").write_text(
            json.dumps(run_meta, indent=2), encoding="utf-8")
        records = receipt["assets"][:1] if pilot else receipt["assets"]
        folds = ("A",) if pilot else tuple(FOLDS)
        conditions = [(b, r, s) for b in BLOCKS for r in RATES for s in SEEDS]
        if pilot:
            conditions = conditions[:2]
        for asset_index, record in enumerate(records, start=1):
            ticker = record["ticker"]
            frame = load_verified_asset(archive, record)
            for fold in folds:
                shard_path = shard_dir / f"{ticker}_{fold}.csv"
                meta_path = shard_dir / f"{ticker}_{fold}.meta.json"
                expected_rows = len(conditions) * len(METHODS) * len(FEATURES)
                if shard_path.exists() and meta_path.exists():
                    sidecar = json.loads(meta_path.read_text())
                    if (sidecar.get("run_fingerprint") == run_fingerprint and
                            sidecar.get("expected_rows") == expected_rows and
                            sidecar.get("csv_sha256") == sha256(shard_path.read_bytes())):
                        print(f"[{asset_index}/{len(records)}] {ticker}/{fold}: verified checkpoint")
                        continue
                partitions = split_frame(frame, fold)
                train = partitions["train"].to_numpy(float)
                test = partitions["test"].to_numpy(float)
                rows = []
                for condition_index, (block, rate, repetition_seed) in enumerate(conditions, 1):
                    train_seed = stable_mask_seed(manifest_sha, ticker, fold, "train",
                                                  block, rate, repetition_seed)
                    test_seed = stable_mask_seed(manifest_sha, ticker, fold, "test",
                                                 block, rate, repetition_seed)
                    train_hidden = strict_block_mask(train.shape, rate, block,
                                                     train_seed, EDGE_GUARD)
                    test_hidden = strict_block_mask(test.shape, rate, block,
                                                    test_seed, EDGE_GUARD)
                    mean, scale = train_observed_scaler(train, train_hidden)
                    train_scaled = (train - mean) / scale
                    test_scaled = (test - mean) / scale
                    estimates = run_methods(corrupt(train_scaled, train_hidden),
                                            corrupt(test_scaled, test_hidden),
                                            repetition_seed)
                    lengths = run_lengths(test_hidden)
                    common = {
                        "ticker": ticker, "country": record["country"],
                        "region": record["region"], "fold": fold,
                        "test_start_year": FOLDS[fold]["test"][0],
                        "test_end_year": FOLDS[fold]["test"][1],
                        "block_length": block, "rate": rate,
                        "seed": repetition_seed,
                        "train_mask_seed": train_seed,
                        "test_mask_seed": test_seed,
                        "actual_max_test_run": int(lengths.max()),
                    }
                    if common["actual_max_test_run"] > block:
                        raise AssertionError("mask-length invariant violated")
                    for method, (estimate, elapsed, warning_count) in estimates.items():
                        rows.extend(score_features(test, estimate, test_hidden,
                                                   mean, scale,
                                                   {**common, "method": method},
                                                   elapsed, warning_count))
                    if condition_index % 10 == 0 or condition_index == len(conditions):
                        print(f"[{asset_index}/{len(records)}] {ticker}/{fold}: "
                              f"{condition_index}/{len(conditions)} conditions")
                if len(rows) != expected_rows:
                    raise AssertionError("missing or duplicate method-feature rows")
                temporary = shard_path.with_suffix(".csv.tmp")
                pd.DataFrame(rows).to_csv(temporary, index=False, float_format="%.17g")
                os.replace(temporary, shard_path)
                sidecar = {
                    "run_fingerprint": run_fingerprint,
                    "expected_rows": expected_rows,
                    "csv_sha256": sha256(shard_path.read_bytes()),
                    "pilot_not_publishable": pilot,
                }
                meta_path.write_text(json.dumps(sidecar, indent=2), encoding="utf-8")
        expected_shards = len(records) * len(folds)
        verified_shards = list(shard_dir.glob("*.meta.json"))
        if len(verified_shards) < expected_shards:
            raise AssertionError("run ended with missing checkpoints")
        (output_dir / "COMPLETE.marker").write_text(
            json.dumps({"expected_shards": expected_shards,
                        "expected_rows": expected_shards * len(conditions)
                        * len(METHODS) * len(FEATURES),
                        "pilot_not_publishable": pilot,
                        "run_fingerprint": run_fingerprint}, indent=2),
            encoding="utf-8")
        print("COMPLETE", output_dir)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--snapshot", type=Path, required=True,
                        help="Private data snapshot ZIP from acquisition notebook")
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--pilot", action="store_true",
                        help="Smoke test only: first asset/fold/two masks; not publishable")
    args = parser.parse_args()
    run(args.snapshot, args.out, pilot=args.pilot)


In [ ]:
%%writefile /content/itri_classical/PROTOCOL_v1.md
# ITRI external-validation protocol v1 (locked before new scores)

Locked: 2026-09-23. The asset universe is `asset_universe_v1.json` (34
US-listed country-exposure ETFs; no result-based substitutions). This protocol
is distinct from the archived BIST 30/S&P 500 study. A run that deviates from
this file must receive a new version and be labeled exploratory.

## Scientific question and scope

Does the previously selected ITRI configuration (`w=1`, `p=5`) retain useful
reconstruction accuracy on additional country-exposure ETF time series and
chronologically distinct test periods? This is **offline imputation**, which
may use observations after the missing day. It is not a prospective trading
or causal study. The ETFs are not independent draws from a population of
markets, and their USD prices embed exchange-rate and US-trading-hour effects.

## Input and eligibility

- Request daily, dividend/split-adjusted OHLCV from Yahoo Finance through a
  pinned `yfinance` version for 2015-01-01 (inclusive) to 2026-01-01
  (exclusive). Record retrieval UTC time, library versions, provider response,
  raw CSV SHA-256, cleaned CSV SHA-256, and all exclusions. Do not publicly
  redistribute raw vendor data without a rights check.
- Eligibility is determined **without running ITRI**: at least 95% of the
  expected annual US exchange trading sessions from 2015–2025 in each year;
  all five OHLCV columns present; strictly increasing unique dates; positive
  finite OHLC prices, nonnegative finite volume; no more than five consecutive
  absent ETF trading sessions. These thresholds are intentionally strict. A
  failed symbol is excluded and logged, with no replacement.
- Do not fill native missing rows from Yahoo with unreported interpolation.
  The actual observed-session intersection within each ETF is retained. All
  engineered features, if used, are computed from past and present rows only;
  warm-up rows are removed before masking.

## Features, partition and preprocessing

- **Primary panel:** adjusted open, high, low, close, volume (5 features).
  This avoids giving cross-feature methods deterministic technical indicators
  computed from hidden raw prices. The historic 14-feature experiment remains
  a separate, labeled development result. A 14-feature external analysis is
  secondary and must acknowledge algebraic relationships among features.
- Three expanding-origin folds, per ETF, with nonoverlapping tests:
  A: train 2015–2018, validation 2019, test 2020–2021;
  B: train 2015–2020, validation 2021, test 2022–2023;
  C: train 2015–2022, validation 2023, test 2024–2025.
  Train/validation/test are not randomly shuffled. The 2020–2021 pandemic
  period and subsequent periods are retained, not selected post hoc.
- Fit each feature's location/scale from the **observed cells of the masked
  training partition only**, then apply to validation/test. Report any zero
  scale and a prespecified deterministic fallback. Training-hidden truth may
  be used as a supervision target by methods designed for supervised masking,
  but not to calculate scaling or input features for a masked case.
- ITRI remains exactly `w=1`, `p=5` for the primary external test. These
  settings were chosen on the old two-index development study. No new ETF test
  result may alter them. If a genuine algorithmic extension is developed,
  tune it only on the original two indices or an explicitly separate
  development subset and freeze it before the 34-ETF external test.

## Missingness and comparator fairness

- Main artificial-missingness mechanism: `strict_block_mask`, independently
  within each feature. Missing-cell fractions are 0.10, 0.20, 0.30 of eligible
  cells. Requested maximal contiguous run lengths are 1, 2, 3, 5, 10, 20
  trading days, and fixed mask seeds are 11, 23, 37. Edge guard = 20 rows.
  A single final shorter run may occur when the target count is not divisible
  by run length. Log the *actual* run-length histogram; assert no run exceeds
  the requested length. The same train/validation/test masks and truth are
  given to every comparator at a matching asset/fold/condition.
  Implementation clarification (2026-09-23, before complete external results):
  reserve one full run in each feature before sampling the remaining runs.
  This keeps the prespecified five-feature macro endpoint defined in every
  condition while retaining the exact total masked-cell count, fixed seeds,
  requested maximal run length, and at most one final shorter run. The first
  incomplete Colab attempt stopped on an unmasked feature and yielded no
  complete external-result set; its partial work must not be reported.
- The historic mask generator is analyzed only as a separately labeled
  sensitivity study because adjacent sampled segments can make actual runs
  longer than their nominal length.
- Minimum comparators: linear interpolation and PCHIP (deterministic temporal);
  KNN and chained/iterative imputation (multivariate statistical); BRITS and
  SAITS (dedicated learned temporal). Model implementation, context length,
  augmentation, training epochs, stopping, seed, and hyperparameter budget
  must be frozen **before** their external test scores are inspected.
  Methods are compared under both equal available information and clearly
  reported computational budgets; impossible perfect equality is not claimed.
- Add observed-cell preservation and no-future-data checks. A bidirectional
  offline imputer may use future values *within its own test segment*, but may
  not use another asset's labels or future periods for weight fitting.

## Metrics and unit of analysis

- Compute errors only on artificially hidden test cells. Predeclare primary
  endpoint as macro-average normalized RMSE across five features, where each
  feature RMSE is divided by its **training-observed** standard deviation.
  This avoids treating high-volume features as numerically dominant and keeps
  the denominator independent of the test mask. Also report standardized MAE,
  raw-feature RMSE/MAE, actual hidden-cell counts, run-length strata, and wall
  time/hardware. Near-zero training standard deviation is an eligibility/QC
  failure, not a silent denominator clamp.
- Aggregate three mask seeds within each asset × outer test period ×
  rate × run-length condition. Compare paired methods on those units. For
  the main summary, first average conditions within each asset × period,
  then show **each asset's** three period scores/differences and a distribution
  across the 34 assets. Do not treat 34 × 3 × 18 × 3 rows as independent market
  replicates. Shared shocks can correlate different ETFs; show region-wise
  and leave-one-region-out sensitivity. Any inferential p-value requires a
  dependence-aware rationale; otherwise use effect sizes and descriptive
  uncertainty, not pseudo-precise significance.
- Report all prespecified symbols including QC exclusions. No selective
  removal for poor ITRI performance. Include negative results and failure
  cases. Distinguish original-development results from new external results.

## Reproducibility and publication gate

- A Colab `Run all` release must download inputs (subject to source terms),
  verify/cache hashes, run QC and all methods, checkpoint progress, verify
  row counts/invariants, then create and download a result ZIP automatically.
  It must never silently fall back to a different data date/provider or
  silently omit failed runs. Publish a pinned environment, source commit,
  protocol version, manifest, raw-input hashes, exclusion log, row-level
  output, aggregation scripts, and figure-generation scripts.
- The first-stage acquisition notebook downloads a public-safe QC receipt
  **and a separate private data snapshot**. Keep the private ZIP for exact
  reruns; never add it to the public repository without confirming the data
  vendor's redistribution terms.
- Manuscript updates happen only after complete outputs and table-level
  numerical reconciliation. A broad benchmark alone does not establish
  algorithmic novelty. Journal-specific scope, subscription-route fees,
  current SCIE inclusion, and JCR Q1 must be checked at submission. Acceptance
  cannot be guaranteed.


## Upload the private data archive

When the next cell runs, Colab shows Choose Files. Select the original private ZIP you sent me. Upload one ZIP only. The experiment continues automatically after the upload.

In [ ]:
import hashlib
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one private data ZIP.")
filename, archive_bytes = next(iter(uploaded.items()))
if not filename.lower().endswith(".zip"):
    raise ValueError("The selected file must be a ZIP archive.")
actual_sha256 = hashlib.sha256(archive_bytes).hexdigest()
if actual_sha256 != '48dc63654f63db3ef9d6d5db7ab69ad502aa079e9f6deb4b276cf9b1170f324b':
    raise ValueError("Archive SHA-256 differs from the audited snapshot. "
                     "Upload the original private ZIP.")
snapshot_path = work_dir / "PRIVATE_vendor_snapshot.zip"
snapshot_path.write_bytes(archive_bytes)
del archive_bytes, uploaded
print("Private snapshot verified:", filename)


In [ ]:
import importlib
import sys

result_dir = work_dir / "full_classical_results"
if not snapshot_path.is_file():
    raise FileNotFoundError(f"Uploaded data archive not found: {snapshot_path}")
sys.path.insert(0, str(work_dir))
importlib.invalidate_caches()
import masking
importlib.reload(masking)
import run_classical_benchmark
importlib.reload(run_classical_benchmark)
run_classical_benchmark.run(snapshot_path, result_dir)


In [ ]:
import json
import zipfile

marker_path = result_dir / "COMPLETE.marker"
if not marker_path.exists():
    raise RuntimeError("The full experiment is incomplete.")
marker = json.loads(marker_path.read_text())
shards = sorted((result_dir / "shards").glob("*.csv"))
row_count = sum(sum(1 for _ in path.open(encoding="utf-8")) - 1
                for path in shards)
if len(shards) != marker["expected_shards"] or row_count != marker["expected_rows"]:
    raise RuntimeError("Result shard count/row count did not reconcile.")
result_zip = Path("/content/ITRI_classical_full_results.zip")
with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as out:
    for path in result_dir.rglob("*"):
        if path.is_file():
            out.write(path, "results/" + str(path.relative_to(result_dir)))
    for name in ("masking.py", "run_classical_benchmark.py", "PROTOCOL_v1.md"):
        out.write(work_dir / name, "code_and_protocol/" + name)
    with zipfile.ZipFile(snapshot_path) as private:
        for name in ("asset_universe_v1.json", "acquisition_qc_receipt.json"):
            out.writestr("input_metadata/" + name, private.read(name))
print(f"Complete: {len(shards)} shards, {row_count} feature rows.")
print("Downloading:", result_zip.name)
files.download(str(result_zip))
